In [1]:
import os
import torch
import numpy as np
import xarray as xr
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

from src.dataset import LazyWeatherDataset
from src.preprocessing import flatten_target_dataset, standardize_with_stats, compute_overall_from_daily_stats
from src.models import get_model, get_model_input_dims

In [2]:

# ---------------------------
# IG routine
# ---------------------------

def integrated_gradients(model, x, baseline, scalar_fn, steps=32):
    """
    x, baseline: [1, C, H, W, T]
    returns IG tensor same shape
    """
    alphas = torch.linspace(0, 1, steps, device=x.device).view(-1, 1, 1, 1, 1, 1)
    x_expand = x.expand(steps, *x.shape[1:])
    base_expand = baseline.expand_as(x_expand)

    path = base_expand + alphas * (x_expand - base_expand)
    path.requires_grad_(True)

    outputs = scalar_fn(model(path))  # [steps]
    grads = torch.autograd.grad(outputs.sum(), path)[0]

    avg_grads = grads.mean(dim=0, keepdim=True)
    ig = (x - baseline) * avg_grads
    return ig.detach()


# ---------------------------
# SHASH scalar targets
# ---------------------------

def shash_scalars(pred_params, target_idx):
    """
    pred_params: [B, K*4]
    returns dict of scalar tensors [B]
    """
    K = pred_params.shape[1] // 4
    params = pred_params.view(-1, K, 4)

    mu    = params[:, :, 0]
    sigma = torch.exp(params[:, :, 1])

    mu_t = mu[:, target_idx]
    sig_t = sigma[:, target_idx]

    prob_hi = 1 - 0.5 * (1 + torch.erf((2 - mu_t) / (sig_t * np.sqrt(2))))
    prob_lo = 0.5 * (1 + torch.erf((-2 - mu_t) / (sig_t * np.sqrt(2))))

    return {
        "mean": mu_t,
        "sigma": sig_t,
        "p_gt_2": prob_hi,
        "p_lt_-2": prob_lo,
    }


# ---------------------------
# Baseline builder
# ---------------------------

def build_climatology(loader, device):
    total = None
    count = 0

    for xb, _ in loader:
        xb = xb.to(device)
        if total is None:
            total = xb.sum(dim=0, keepdim=True)
        else:
            total += xb.sum(dim=0, keepdim=True)
        count += xb.shape[0]

    return total / count  # [1, C, H, W, T]


# ---------------------------
# Main driver
# ---------------------------

def run_ig_for_model(model_name, level, inputs, targets, stats,
                     batch_size=32, steps=32, latest=False, n_splits=5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    days = inputs.day.values
    kf = KFold(n_splits=n_splits, shuffle=False)

    overall_stats = compute_overall_from_daily_stats(stats)
    input_dims = get_model_input_dims(model_name)

    for fold, (train_idx, val_idx) in enumerate(kf.split(days)):

        print(f"\n=== Fold {fold} ===")

        train_days = days[train_idx]
        val_days = days[val_idx]

        X_train = inputs.sel(day=train_days)
        X_val = inputs.sel(day=val_days)

        fold_stats = compute_overall_from_daily_stats(stats.sel(day=train_days))

        conversion_stats = xr.Dataset({
            v: ((fold_stats[v] - overall_stats[v]) / overall_stats[v.replace('_mean', '_std')])
            if v.endswith('_mean')
            else (fold_stats[v] / overall_stats[v])
            for v in fold_stats.data_vars
        })

        X_train_std = standardize_with_stats(X_train, conversion_stats)
        X_val_std   = standardize_with_stats(X_val, conversion_stats)

        train_ds = LazyWeatherDataset(
            X_train_std,
            y=flatten_target_dataset(targets.sel(time=train_days)),
            input_dimensions=5
        )

        val_ds = LazyWeatherDataset(
            X_val_std,
            y=flatten_target_dataset(targets.sel(time=val_days)),
            input_dimensions=5
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size)
        val_loader   = DataLoader(val_ds, batch_size=1)

        # ---- Build baseline ----
        print("Building baseline climatology...")
        baseline = build_climatology(train_loader, device)

        # ---- Load model ----
        model = get_model(model_name, next(iter(train_loader))[0].shape[1:], 36, targets=None).to(device)

        model_path = f"models/{model_name}/level={level}/fold={fold}/" + ("latest.pt" if latest else "best.pt")
        model.load_state_dict(torch.load(model_path, map_location="cpu")["model_state_dict"])
        model.eval()

        # ---- Accumulators ----
        C, H, W, T = baseline.shape[1:]
        K = 9
        scalars = ["mean", "sigma", "p_gt_2", "p_lt_-2"]

        chan_acc = {s: torch.zeros(K, C) for s in scalars}
        spat_acc = {s: torch.zeros(K, H, W) for s in scalars}
        temp_acc = {s: torch.zeros(K, T) for s in scalars}

        n_days = 0

        # ---- IG loop ----
        for xb, _ in val_loader:
            xb = xb.to(device)
            n_days += 1

            for target_idx in range(K):

                def scalar_fn(out):
                    return shash_scalars(out, target_idx)[scalar_name]

                for scalar_name in scalars:
                    ig = integrated_gradients(model, xb, baseline, scalar_fn, steps=steps)[0]

                    chan_acc[scalar_name][target_idx] += ig.abs().sum(dim=(1,2,3))
                    spat_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0,3))
                    temp_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0,1,2))

        # ---- Normalize ----
        for s in scalars:
            chan_acc[s] /= n_days
            spat_acc[s] /= n_days
            temp_acc[s] /= n_days

        # ---- Save ----
        out_dir = f"results/ig/{model_name}/fold_{fold}"
        os.makedirs(out_dir, exist_ok=True)

        torch.save({
            "channel": chan_acc,
            "spatial": spat_acc,
            "temporal": temp_acc
        }, os.path.join(out_dir, "ig_results.pt"))

        print(f"Saved fold {fold}")

In [ ]:
inputs = xr.open_zarr("/glade/work/milesep/convective_outlook_ml/train_inputs_slgt_small_glade.zarr")
targets = xr.open_dataset("data/processed_data/train_targets_slgt_new.nc")
stats = xr.open_dataset("data/processed_data/daily_input_stats_slgt_small_glade.nc")

In [ ]:
run_ig_for_model(
    model_name='cnn3d_gelu_0_5/level=slgt_small_glade_new/opt=Adam_lr=0.001_batch=8_crit=ShashNLL',
    level='slgt_small_glade_new',
    inputs=inputs,
    targets=targets,
    stats=stats,
    steps=32,
)